# kNN & Distance-Based Learning — AI Lab Instructor Notebook

*Classical ML · Medium*

Build strong intuition for kNN, distance metrics, curse of dimensionality, and bias–variance tradeoffs.

**Outcome.** Students can implement kNN from scratch (vectorized), reason about metric choice, and explain failure modes at FAANG interview depth.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# kNN & Distance-Based Learning — Student Lab

Complete all TODOs. Focus: vectorization, scaling, and failure modes.

## Section 0 — Synthetic dataset generator
We’ll generate Gaussian blobs with controllable dimension.

<img src="/images/w3-d4-blobs-sample-xs.png" alt="Sample Gaussian Blobs (d=2)" width="1200" />

In [ ]:
$$10

**Why this works.** # kNN & Distance-Based Learning — Instructor (Solutions + Rationale)

**Sample Image with d = 2**


**Distance Concentration (intuition image)**

## Task 2: L2 distance matrix
*Section: Vectorized distances*

## Section 1 — Vectorized distances

### Task 1.1: L2 distance matrix
Compute D where D[i,j] = ||X_test[i] - X_train[j]||_2.

- Use expansion: ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
- Avoid building (m,n,d) broadcast tensor for large sizes

**Hints**
- Use expansion: ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
Avoid building (m,n,d) broadcast tensor for large sizes

In [ ]:
def l2_distances(X_test, X_train):
    # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
    a2 = np.sum(X_test*X_test, axis=1, keepdims=True)
    b2 = np.sum(X_train*X_train, axis=1, keepdims=True).T
    dist2 = a2 + b2 - 2*(X_test @ X_train.T)
    dist2 = np.maximum(dist2, 0.0)
    return np.sqrt(dist2)

D = l2_distances(X_test[:10], X_train[:20])
check('D_shape', D.shape == (10,20))

# Rationale: axis 0 = ↓, axis 1 = →;
# Rationale: dot-product expansion avoids allocating (m,n,d) broadcast tensors (memory win).



In [ ]:
# Checks
check('D_shape', D.shape == (10,20))

**Why this works.** axis 0 = ↓, axis 1 = →;

## Task 3: L1 distances (optional)
*Section: Vectorized distances*

### Task 1.2: L1 distances (optional)
Compute L1 distance matrix (can use broadcasting here since we’ll keep sizes small).

**Interview Angle:** When might L1 beat L2?

In [ ]:
def l1_distances(X_test, X_train):
    return np.sum(np.abs(X_test[:, None, :] - X_train[None, :, :]), axis=2)

D1 = l1_distances(X_test[:5], X_train[:7])
check('D1_shape', D1.shape == (5,7))

# Rationale: L1 can be more robust to outliers; geometry differs vs L2.  
# Rationale: axis 0 -> test samples, axis 1 -> train samples, axis 2 -> feature dimensions.


In [ ]:
# Checks
check('D1_shape', D1.shape == (5,7))

**Why this works.** L1 can be more robust to outliers; geometry differs vs L2.

knn X_test[:, `None`, :]  while calculating Manhattan distance `None` is added so that Each test point becomes a “stack” of 1 row, waiting to be compared with all test points.

## Task 4: Predict with kNN
*Section: kNN classifier*

## Section 2 — kNN classifier

### Task 2.1: Predict with kNN

- compute distances
- argsort distances to find k nearest
- majority vote

**FAANG gotcha:** define tie-breaking deterministically (e.g., pick smallest label).

**Hints**
- compute distances
argsort distances to find k nearest
majority vote

In [ ]:
def knn_predict(X_train, y_train, X_test, k=3):
    D = l2_distances(X_test, X_train)
    nn_idx = np.argsort(D, axis=1)[:, :k]
    nn_labels = y_train[nn_idx]
    votes = nn_labels.sum(axis=1)
    # deterministic tie-break: for even k, choose class 0 on tie
    return (votes*2 > k).astype(int)

yhat = knn_predict(X_train, y_train, X_test[:20], k=3)
check('yhat_shape', yhat.shape == (20,))
check('labels', set(np.unique(yhat)).issubset({0,1}))

# Rationale: small k -> low bias/high variance; large k -> higher bias/lower variance.

In [ ]:
# Checks
check('yhat_shape', yhat.shape == (20,))
check('labels', set(np.unique(yhat)).issubset({0,1}))

**Why this works.** small k -> low bias/high variance; large k -> higher bias/lower variance.

## Task 5: Evaluate over k
*Section: kNN classifier*

### Task 2.2: Evaluate over k
Compute accuracy for k in [1,3,5,9,15].

**Checkpoint:** Why does increasing k usually increase bias and reduce variance?

In [ ]:
def accuracy(y, yhat):
    return float(np.mean(y == yhat))

ks=[1,3,5,9,15]
for k in ks:
    tr = accuracy(y_train, knn_predict(X_train, y_train, X_train, k))
    te = accuracy(y_test, knn_predict(X_train, y_train, X_test, k))
    print('k', k, 'train', tr, 'test', te)

## Task 6: Distance concentration
*Section: Curse of dimensionality*

## Section 3 — Curse of dimensionality

### Task 3.1: Distance concentration
For increasing dimension d, compute ratio min_dist/max_dist for random points and show it approaches 1.

- sample n points in d dims
- compute pairwise distances from one reference point

![Distance Concentration L2](/images/w3-d4-distance-concentration-l2.png)

**Hints**
- sample n points in d dims
compute pairwise distances from one reference point

In [ ]:
def concentration_ratio(n=2000, dims=(2,5,10,50,100)):
    ratios=[]
    for d in dims:
        X = rng.standard_normal((n,d))
        ref = X[0:1]
        D = l2_distances(ref, X)[0,1:]
        ratios.append((d, float(D.min()/D.max())))
    return ratios

for d, r in concentration_ratio():
    print('d', d, 'min/max', r)

# Rationale: as d increases, distances concentrate -> nearest vs farthest become similar.

**Why this works.** as d increases, distances concentrate -> nearest vs farthest become similar.

## Task 7: Break kNN with scaling
*Section: Feature scaling sensitivity*

## Section 4 — Feature scaling sensitivity

### Task 4.1: Break kNN with scaling
Create a dataset where one feature has huge scale and show accuracy drops. Then fix with standardization.

In [ ]:
# Scaling sensitivity demo
Xtr, ytr, Xte, yte = make_blobs(d=2)
Xtr_bad = Xtr.copy(); Xte_bad = Xte.copy()
Xtr_bad[:,1] *= 1000; Xte_bad[:,1] *= 1000
acc_bad = accuracy(yte, knn_predict(Xtr_bad, ytr, Xte_bad, k=5))

mu = Xtr_bad.mean(axis=0, keepdims=True)
sd = Xtr_bad.std(axis=0, keepdims=True) + 1e-12
Xtr_std = (Xtr_bad - mu) / sd
Xte_std = (Xte_bad - mu) / sd
acc_std = accuracy(yte, knn_predict(Xtr_std, ytr, Xte_std, k=5))

print('acc_bad', acc_bad, 'acc_std', acc_std)
check('improves', acc_std >= acc_bad)

# Rationale: kNN relies on distance; feature scaling changes distance geometry drastically.

In [ ]:
# Checks
check('improves', acc_std >= acc_bad)

**Why this works.** kNN relies on distance; feature scaling changes distance geometry drastically.